In [ ]:
import sys

print(sys.executable)

# Import des données et description générale

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as ss

In [ ]:
df = pd.read_csv('../data/raw/avis_40k.csv', sep=';', header=0)
print("chargement des données et description rapide")

display(df.head())
display(df.info())
display(df.describe())

# Définition du problème

* Avant toute chose il est bon de définir **Quelle est la cible ?**. Pour ce projet, l'objectif est de catégoriser les avis utilisateurs de manière automatique, c'est donc un problème de clustering. Il n'y a pas de variable cible comme dans un cas classique de classification supervisée. Il faudra néanmoins réfléchir par la suite aux indicateurs pertinents pouvant être utilisés pour estimer la qualité des clusters produits.
* La variable utilisée pour ce clustering est la variable **commentaire** correspondant aux avis utilisateurs. Ces avis sont issus d'un scrapping de TrustePilot pour 40k avis concernant l'entreprise Oscaro. Les autres variables seront utiliser pour l'exploration des données.

# Traitement des NA et suppression des variables inutiles

In [ ]:
print('traitement des NaN')

print(df['commentaire'].isna().sum())
print('pourcentage Na parmi les avis:', df['commentaire'].isna().sum()/len(df['commentaire'])*100)
print(df[df['commentaire'].isna()].head())


plt.figure(figsize=(10,10))
plt.pie(df["commentaire"].isna().value_counts(normalize=True),
            labels=['avis', 'NaN'],
            explode=[0.1, 0.1],
            autopct="%0.2f%%",
            colors=["palevioletred", "grey"])
plt.title('Proportion de NaN parmi les avis clients')

In [ ]:
df['commentaire'] = df['commentaire'].fillna(df['titre']) # remplacement na par titre
df[['commentaire', 'auteur', 'date_avis']] = df[['commentaire', 'auteur', 'date_avis']].drop_duplicates() # suppression des duplicats
df = df.drop('date_avis', axis=1) # suppression date avis, on garde uniquement date experience
df = df.drop('texteServiceReply', axis=1) # suppression de la variable car n'apporte pas d'info
df = df.drop('url', axis = 1)
df = df.drop('dateServiceReply', axis = 1)

display(df.info())
display(df.head())

# Traitement de la langue dans le corpus d'avis

In [ ]:
from langdetect import detect
from langdetect import DetectorFactory
DetectorFactory.seed = 34

def to_detect(text_i): # détection de la langue
    try :
        return detect(text_i)
    except:
        return ('NA')

df["lang"] = df["commentaire"].apply(lambda x: to_detect(x))
print(df['lang'].value_counts())
print(df['lang'].describe(include='O'))

In [ ]:
display(df.info())
display(df.head())

In [ ]:
df = df[df["lang"] == "fr"] # on conserve uniquement les avis en français
df = df.dropna(subset=['commentaire'])
df.head()
df.info()

# Création de nouvelles variables pour la visualisation temporelle

In [ ]:
# Trier chronologiquement les avis
df["date_to_sort"] = pd.to_datetime(
    df["date_experience"],
    format="%B %d, %Y",
)

df = df.sort_values("date_to_sort").reset_index(drop=True)

# Entraînement initial : 60 %
df_train = df.iloc[:24000].copy()

# Référence du monitoring : 10 %
df_reference = df.iloc[24000:28000].copy()

# Production simulée : 20 %
df_production = df.iloc[28000:36000].copy()

# Évaluation finale : 10 %
df_test = df.iloc[36000:].copy()


In [ ]:
month_to_number = {'January' : 1, 'February' : 2, 'March' : 3, 'April' : 4, 'May' : 5, 'June' : 6, 'July' : 7, 'August' : 8, 'September' : 9, 'October' : 10, 'November' : 11, 'December' : 12}

df['date_experience_year'] = df['date_experience'].str[-4:] # transformation variable date_experience en 2 variables: année et mois
df['date_experience_month'] = df['date_experience'].replace(month_to_number, regex=True).astype(int)
df['date_experience_year'] = df['date_experience_year'].astype(int)
df = df.drop('date_experience', axis=1)

# création comm_reply : fusion avis + réponse entreprise
df['comm_reply'] = np.where(df['dataServiceReply'].isna() == False, df['commentaire'] + df['dataServiceReply'], df['commentaire'])

# création length_comm : taille de l'avis
df['length_comm'] = df['commentaire'].apply(lambda x: len(x))

plt.figure(figsize=(20,10))

plt.subplot(121)

plt.pie(df["date_experience_year"].value_counts(normalize=True),
            labels=['2025', '2026', '2024'],
            explode=[0.1, 0.1, 0.1],
            autopct="%0.2f%%",
            colors=["palevioletred", "darksalmon", "grey"])
plt.title('Représentation des années dans le dataset')

plt.subplot(122)

plt.pie(df["date_experience_month"].value_counts(normalize=True),
            labels=['mai', 'novembre', 'aout', 'juin', 'octobre', 'juillet', 'décembre', 'janvier', 'septembre', 'avril', 'février', 'mars'],
            #explode=[0.1, 0.1, 0.1],
            autopct="%0.2f%%")
            #colors=["palevioletred", "darksalmon", "grey"])
plt.title('Représentation des mois dans le dataset')


df["date_experience_month"].value_counts(normalize=True)

# Distribution des notes et de la taille des commentaires

In [ ]:
plt.figure(figsize=(20,10))

plt.subplot(131)

sns.histplot(df['note'])
plt.title('Distribution des notes')

plt.subplot(132)

sns.histplot(df['length_comm'])
plt.title('Distribution de la taille des commentaires')

plt.subplot(133)

sns.barplot(data= df[['note','length_comm']].groupby('note').median(), x='note', y='length_comm')
plt.title('taille médiane des commentaires en fonction de la note')

In [ ]:
plt.figure(figsize=(15,10))
plt.subplot(1,2,1)
sns.scatterplot(x=df['length_comm'], y=df['note'])
plt.title('Distribution des notes en fonction de la taille des commentaires')
plt.subplot(122)
ss.probplot(df['length_comm'], plot=plt)
plt.title('QQ-plot de la taille des commentaires')
plt.show()



#plt.subplot(223)
#sns.lmplot(x='length_comm', y='note', data=df, )

print(df[['note', 'length_comm']].corr())

print('normalité des données, shapiro\'s p.value:', ss.shapiro(df['length_comm'].head(5000))[1])
print('pearson correlation coef:', ss.pearsonr(df['length_comm'], df['note'])[0])
print('pearson correlation p.value:', ss.pearsonr(df['length_comm'], df['note'])[1])
print('kendall correlation coef:', ss.kendalltau(df['length_comm'], df['note'])[0])
print('kendall correlation p.value:', ss.kendalltau(df['length_comm'], df['note'])[1])

# Export des données

In [ ]:
df_train.to_csv('../data/raw/avis_24k_train.csv') # entrainement initial
df_reference.to_csv('../data/raw/avis_4k_reference.csv') # référence monitoring
df_production.to_csv('../data/raw/avis_8k_production.csv') # production simulée
df_test.to_csv('../data/raw/avis_3k_test.csv') # test final

# Wordcloud

In [ ]:
import spacy
from collections import Counter
from wordcloud import WordCloud

# Chargement du modèle linguistique français
nlp = spacy.load("fr_core_news_sm")

# Stopwords supplémentaires, adaptés au corpus
custom_stopwords = {
    "être", "avoir", "faire", "dire",
    "aller", "voir", "pouvoir"
}

def preprocess_text(text):
    """
    Nettoie et lemmatise un texte français.
    Retourne une liste de tokens.
    """
    doc = nlp(text.lower())

    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
        and not token.like_url
        and not token.like_email
        and token.is_alpha
        and len(token.lemma_) > 2
        and token.lemma_.lower() not in custom_stopwords
    ]

    return tokens


# Exemple de corpus
texts = df[df['note']==1]['commentaire']
# Prétraitement de tous les documents
tokens = []

for text in texts:
    tokens.extend(preprocess_text(text))

# Calcul des fréquences
frequencies = Counter(tokens)

# Génération du WordCloud
wc = WordCloud(
    width=1200,
    height=600,
    background_color="white",
    max_words=100,
    collocations=False,
    random_state=42
).generate_from_frequencies(frequencies)

# Visualisation
plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud du corpus français", fontsize=16)
plt.show()